# 01 · Prepare data (run once per platform)

Gets FFHQ-256 (70k aligned faces) and COCO val2017 (5k scenes), normalises
everything to 256 px, detects 5-point face landmarks, fixes the held-out eval
split (300 FFHQ + 100 COCO, seed 42; identical on every platform), and stores
the result as one `prepared.tar`. Later notebooks only unpack that tar.
About 30–45 minutes on a T4.

**Colab:** the cell asks for `kaggle.json` (kaggle.com → Settings → API →
Create New Token) to download FFHQ. The tar goes to Drive.

**Kaggle:** first **Add Input** → search `flickrfaceshq-dataset-nvidia-resized-256px`
(by xhlulu), so nothing is downloaded. Then **Save Version → Save & Run All**.
When it finishes, that version's output holds `soteria/data/prepared.tar`:
attach it to 02 and 03 with **Add Input → Your Work** (this notebook's output).

**Colab:** results go to Google Drive (`MyDrive/soteria`). Runtime → T4 GPU.

**Kaggle:** Settings → Accelerator **GPU T4 x2**, Internet **On** (needs a
phone-verified account). Results go to `/kaggle/working/soteria` and are only
kept when you save a version with **Save Version → Save & Run All**; from an
interactive session, download what you need from the Output panel before it ends.
Secrets (`HF_TOKEN`, `GITHUB_TOKEN` for a private fork) go in Add-ons → Secrets.

In [ ]:
# ---- setup: lets this cell run on its own, on Colab or Kaggle; safe to rerun ----
REPO_URL = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH   = "main"
import importlib, os, subprocess, sys
KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/input")
if not KAGGLE and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")                # Colab: results live on Drive
CODE = "/tmp/Project-Soteria" if KAGGLE else "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
def _secret(name):
    try:
        if KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    _token = _secret("GITHUB_TOKEN")             # only needed for a private fork
    _git("clone", "-q", "-b", BRANCH, REPO_URL.replace("https://", f"https://{_token}@") if _token else REPO_URL, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.runtime as rt; importlib.reload(rt)
PLATFORM = rt.platform()
ROOT, DATA = rt.prepare(data=False, gpu=True, requirements=('requirements.txt', 'requirements-data.txt'))
# ---------------------------------------------------------------------------------

from pathlib import Path
sh = get_ipython().system            # same as "!", but usable inside if-blocks
RAW = Path("/tmp/raw")
if PLATFORM == "kaggle":
    FFHQ = rt.find_input_dir("flickrfaceshq")
    if FFHQ is None:
        raise RuntimeError("Attach FFHQ-256 first: Add Input > search "
                           "'flickrfaceshq-dataset-nvidia-resized-256px' (by xhlulu)")
else:
    FFHQ = RAW / "ffhq"
    if not FFHQ.is_dir():
        from google.colab import files
        if not Path("/root/.kaggle/kaggle.json").exists():
            up = files.upload()                      # choose kaggle.json
            Path("/root/.kaggle").mkdir(exist_ok=True)
            Path("/root/.kaggle/kaggle.json").write_bytes(up["kaggle.json"])
            os.chmod("/root/.kaggle/kaggle.json", 0o600)
        sh(f"kaggle datasets download -d xhlulu/flickrfaceshq-dataset-nvidia-resized-256px -p {FFHQ} --unzip -q")
COCO = RAW / "coco" / "val2017"
if not COCO.is_dir():
    sh(f"mkdir -p {RAW}/coco && wget -q http://images.cocodataset.org/zips/val2017.zip -O {RAW}/coco/val2017.zip"
       f" && unzip -q {RAW}/coco/val2017.zip -d {RAW}/coco && rm {RAW}/coco/val2017.zip")
print("FFHQ:", FFHQ, "| COCO:", COCO)

PREPARED = rt.local_data_dir() / "prepared"
if not (PREPARED / "manifest.csv").exists():
    sh(f"python scripts/prepare_data.py --ffhq {FFHQ} --coco {COCO} --out {PREPARED}")
TAR = ROOT / "data" / "prepared.tar"
TAR.parent.mkdir(parents=True, exist_ok=True)
sh(f"tar -cf {TAR} -C {PREPARED.parent} prepared && ls -lh {TAR}")